# 02 · AlexNet / VGG / NiN on Fashion-MNIST —— 深度加深的时代

**家族位置**：`02_CNN_Family` 第 2 个项目（前：`01_LeNet_MNIST`；后：`03_ResNet_CIFAR10` ★）

**与上一站的衔接**：LeNet 只有 2 个卷积层就能在 MNIST 拿 98.88%。但 MNIST 太简单——灰底白字、居中、无背景干扰。**Fashion-MNIST** 同样 28×28、同样 10 类，难度陡增（T恤/衬衫/外套这类语义相近的类别互相混淆），它逼我们把网络**加深加宽**——这正是 2012-2014 年的真实历史：AlexNet（深度+ReLU+Dropout）→ VGG（深度=小卷积核堆叠）→ NiN（1×1 卷积+全局平均池化）。

**学习目标**
1. 体验"数据变难"直接看错误样本：Fashion-MNIST 的混淆是语义级的（衬衫 vs 外套 vs T恤）
2. 理解深度时代的三个关键组件：**ReLU**（深层梯度流）、**Dropout**（大模型防过拟合）、**小卷积核堆叠**（两层 3×3 = 一层 5×5 感受野，但参数更少、非线性更多）
3. 数据增强首秀：随机翻转/裁剪的"免费扩容"效果
4. 如实面对训练难点：VGG 收敛慢、NiN（无 BN 时代）调参敏感——为 03 的 ResNet 埋伏笔

## 1. 原理：2012-2014 深度时代的三大组件

### ReLU：深度的发动机

Sigmoid/Tanh 在深层会饱和（梯度≈0，01 家族 03 项目实测过 Sigmoid 最慢）。ReLU 负区恒为 0、正区梯度恒为 1，**堆多少层梯度都不衰减**——AlexNet 用它第一次把 8 层网络训通。

### Dropout：大模型的保险

参数量上到百万级，训练集被死记的风险剧增。Dropout 训练时随机关 50% 神经元，逼网络不依赖任何单个神经元（01 家族 03 项目实测 Dropout +0.6pt，这里通道更多、更关键）。

### 小卷积核堆叠：VGG 的核心洞察

两层 3×3 卷积的感受野 = 一层 5×5，但：

| 方案 | 参数量（C 通道） | 非线性次数 |
|---|---|---|
| 一层 5×5 | 25C² | 1 |
| 两层 3×3 | 18C² | 2 |

**参数 -28%、非线性 +100%**——"深度本身就是好的"第一次被系统证明。

### NiN：用 1×1 卷积替代全连接

1×1 卷积 = 在每个像素位置上做全连接（通道间混合）。NiN 用它构建"微型全连接块"，最后用**全局平均池化**直接出分类——全连接参数几乎清零。代价：2013 年没有 BN，深层堆叠对初始化和 lr 敏感（下面会现场见识）。

### 本项目的 28×28 精神复现约定

原版 AlexNet 输入 227×227、VGG 224×224。CPU 预算下保持**块的层数、通道比例、Dropout、池化次数**，缩小分辨率与通道数（模型名带 Mini），参数量 1.3M 量级——架构思想不变。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import FASHION_CLASSES, load_fashion_mnist_torch
from common.engine import fit
from common.models import AlexNetMini, LeNet, NiNMini, VGGMini
from common.utils import count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：Fashion-MNIST

同样 60k+10k、28×28 灰度、10 类（标准化用 0.2860/0.3530）。注意观察样本图——没有 MNIST 的"居中白字"特权，类别之间是真实的语义差异。

In [ ]:
DATA_ROOT = ROOT / "data"
Xtr, ytr, Xte, yte = load_fashion_mnist_torch(str(DATA_ROOT))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
rng = np.random.default_rng(0)
for r in range(2):
    for c in range(10):
        idx = int(rng.integers(len(ytr)))
        axes[r, c].imshow(Xtr[idx, 0], cmap="gray")
        axes[r, c].set_title(FASHION_CLASSES[ytr[idx].item()], fontsize=8)
        axes[r, c].axis("off")
plt.suptitle("Fashion-MNIST 随机样本：类别靠语义区分，不再靠笔形", fontsize=11)
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 实验协议

**受控对照**：四模型（LeNet / AlexNetMini / VGGMini / NiNMini）同数据（20k 训练子集）、同 3 epochs、同 Adam(1e-3)、同 seed=0——唯一变量是架构。全量 10k 测试集统一评估。

> 为什么用子集：原型实验实测（全通道版 2ep@10k 需 87~132s），原版通道全量训练需 40+ 分钟/模型；缩通道后 3ep@20k 约 25~37s/模型，教学对比足够（本项目的看点是**相对差距**，不是绝对 SOTA）。

**参数量与层数一览**（notebook 中打印复核）：

In [ ]:
SUBSET, EPOCHS = 20000, 3
tr = DataLoader(TensorDataset(Xtr[:SUBSET], ytr[:SUBSET]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)

for name, m in [("LeNet", LeNet()), ("AlexNetMini", AlexNetMini()),
                ("VGGMini", VGGMini()), ("NiNMini", NiNMini())]:
    n_conv = sum(1 for mod in m.modules() if isinstance(mod, nn.Conv2d))
    n_fc = sum(1 for mod in m.modules() if isinstance(mod, nn.Linear))
    print(f"{name:12s} conv层数={n_conv:2d} fc层数={n_fc} 参数量={count_params(m):>9,}")

## 4. 主实验：四模型同台

预期（按缩时原型）：AlexNetMini/VGGMini 领先，LeNet 受限于 2 层卷积的表达力，NiNMini 受限于无 BN 的收敛慢。跑完对答案。

In [ ]:
results = {}
for name, cls in [("LeNet", LeNet), ("AlexNetMini", AlexNetMini),
                  ("VGGMini", VGGMini), ("NiNMini", NiNMini)]:
    set_seed(0)
    model = cls()
    hist = fit(model, tr, te, epochs=EPOCHS, lr=1e-3, verbose=False)
    results[name] = hist
    print(f"{name:12s} val_acc={hist['val_acc'][-1]:.2%} | val_loss={hist['val_loss'][-1]:.4f} | train_acc={hist['train_acc'][-1]:.2%}")

print("\n（train_acc 与 val_acc 的差距 = 过拟合信号，第 6 节分析）")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order = ["LeNet", "AlexNetMini", "VGGMini", "NiNMini"]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
for name, c in zip(order, colors):
    axes[0].plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
    axes[1].plot(results[name]["train_acc"], marker="o", ms=4, label=name, color=c)
axes[0].set_title("val_acc（Fashion-MNIST 10k 测试集）")
axes[1].set_title("train_acc（20k 子集）")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig1_models.png", dpi=150, bbox_inches="tight")
plt.show()

accs = {n: results[n]["val_acc"][-1] for n in order}
fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.bar(order, [accs[n] for n in order], color=colors)
for b, n in zip(bars, order):
    ax.text(b.get_x() + b.get_width() / 2, accs[n], f"{accs[n]:.2%}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0.5, 1.0); ax.set_ylabel("val_acc")
ax.set_title(f"四模型同台（{SUBSET//1000}k 子集 · {EPOCHS} epochs · seed=0）")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 数据增强首秀：随机翻转 + 随机平移

增强 = 对训练图做**保标签的随机变换**（衣服左右翻转还是衣服；随机平移不改类别），等效于免费扩充数据集。用 AlexNetMini 做受控对照：唯一变量是有无增强，并跑 3ep / 10ep 两档训练时长——增强的收益未必是"免费的"，它取决于模型有没有把原数据喂饱（3ep 短训下的负结果已在原型阶段实测，如实呈现）。

In [ ]:
def augment_batch(xb: torch.Tensor, g: torch.Generator) -> torch.Tensor:
    """Fashion-MNIST 增强两件套（作用在 (B,1,28,28) 张量上）：
    1) 50% 概率水平翻转（衣服左右翻转保标签）
    2) 4 像素随机平移（padding=4 + 随机裁剪回 28×28）
    随机性全部来自传入的 generator——不依赖全局 RNG，序列可复现。
    """
    if torch.rand(1, generator=g).item() < 0.5:
        xb = torch.flip(xb, dims=[3])
    pad = torch.nn.functional.pad(xb, (4, 4, 4, 4))
    i = torch.randint(0, 9, (1,), generator=g).item()
    j = torch.randint(0, 9, (1,), generator=g).item()
    return pad[:, :, i:i + 28, j:j + 28]


class AugLoader:
    """在 collate 后整批增强（CPU 张量实现，等价 torchvision 随机变换的效果）。

    洗牌与增强共用固定种子的独立 generator——可复现。
    """

    def __init__(self, x, y, batch_size=128, seed=0):
        self.x, self.y, self.bs = x, y, batch_size
        self.g = torch.Generator().manual_seed(seed)

    def __iter__(self):
        n = len(self.x)
        idx = torch.randperm(n, generator=self.g)
        for s in range(0, n, self.bs):
            b = idx[s:s + self.bs]
            yield augment_batch(self.x[b], self.g), self.y[b]

    def __len__(self):
        return (len(self.x) + self.bs - 1) // self.bs


# 对照设计：唯一变量是有无增强；增强组跑 3ep 与 10ep 两档，看收益如何随训练时长变化。
# 两档共用同一增强序列（同 seed），且 10ep 的前 3 轮应与 3ep 完全一致——可复现性的强验证。
set_seed(0)
tr_aug = AugLoader(Xtr[:SUBSET], ytr[:SUBSET], seed=0)
hist_aug3 = fit(AlexNetMini(), tr_aug, te, epochs=3, lr=1e-3, verbose=False)

set_seed(0)
tr_aug = AugLoader(Xtr[:SUBSET], ytr[:SUBSET], seed=0)
hist_aug10 = fit(AlexNetMini(), tr_aug, te, epochs=10, lr=1e-3, verbose=False)

print(f"AlexNetMini 无增强 3ep : val_acc={accs['AlexNetMini']:.2%} | train_acc={results['AlexNetMini']['train_acc'][-1]:.2%}")
print(f"AlexNetMini +增强 3ep  : val_acc={hist_aug3['val_acc'][-1]:.2%} | train_acc={hist_aug3['train_acc'][-1]:.2%}")
print(f"AlexNetMini +增强 10ep : val_acc={hist_aug10['val_acc'][-1]:.2%} | train_acc={hist_aug10['train_acc'][-1]:.2%}")
assert hist_aug3['val_acc'] == hist_aug10['val_acc'][:3], "同 seed 下前 3 轮应完全一致"
print("✔ 10ep 曲线前 3 轮与 3ep 完全一致——增强序列可复现实锤")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
labels = [f"无增强\n{EPOCHS}ep", f"+增强\n3ep", f"+增强\n10ep"]
vals = [accs["AlexNetMini"], hist_aug3["val_acc"][-1], hist_aug10["val_acc"][-1]]
bars = ax.bar(labels, vals, color=["#4C72B0", "#C44E52", "#DD8452"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2%}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0.62, 0.92); ax.set_ylabel("val_acc")
ax.set_title("数据增强对照（AlexNetMini · 20k 子集 · seed=0）")
plt.tight_layout()
plt.savefig(FIGS / "fig3_aug.png", dpi=150, bbox_inches="tight")
plt.show()

delta3 = hist_aug3["val_acc"][-1] - accs["AlexNetMini"]
delta10 = hist_aug10["val_acc"][-1] - accs["AlexNetMini"]
print(f"相对无增强 {EPOCHS}ep 基线：+增强 3ep {delta3:+.2%} | +增强 10ep {delta10:+.2%}")

## 6. 深度难点现场：冠军的错误与 NiN 的困境

### 6.1 冠军模型的混淆矩阵

MNIST 的错误是"笔形相近"（4/9、3/5）；Fashion-MNIST 的错误是**语义相近**——看最常混淆的对是什么类别。

In [ ]:
best_name = max(accs, key=accs.get)
print("冠军:", best_name, f"{accs[best_name]:.2%}")
set_seed(0)
best_model = {"LeNet": LeNet, "AlexNetMini": AlexNetMini, "VGGMini": VGGMini, "NiNMini": NiNMini}[best_name]()
fit(best_model, tr, te, epochs=EPOCHS, lr=1e-3, verbose=False)
best_model.eval()
with torch.no_grad():
    pred = best_model(Xte).argmax(1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(FASHION_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(FASHION_CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=6)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title(f"{best_name} 混淆矩阵（真实类别名）")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIGS / "fig4_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
for k in flat:
    r, c = np.unravel_index(k, cm.shape)
    print(f"  {FASHION_CLASSES[r]:12s} → {FASHION_CLASSES[c]:12s} : {cm_off[r, c]} 次")

### 6.2 NiN 的困境：没有 BN 的年代有多难

NiNMini 在 lr=1e-3 时 3 epochs 只有 ~30%，把 lr 提到 1e-2/3e-2 更是直接崩到 10%（原型实验数据，不重复烧时间）。**原因**：深层 1×1 堆叠 + GAP 头对初始化极敏感，梯度在 9 层堆叠里不稳。这不是实现错误——NiN 原论文（2013）就报告调参敏感，直到 2015 年 BN 出现这类深堆叠才"好训"。

**伏笔**：03 项目 ResNet 的两条腿——BN（稳定每层输入分布）+ 残差（给梯度留直达通道）——正是治这两个病的药。

## 7. 总结与下一步

**本项目收获**

1. 数据升级（MNIST→Fashion-MNIST）立刻暴露 MLP/LeNet 的表达力上限——加深加宽是必然
2. 深度时代三件套到位：ReLU（梯度流）、Dropout（防死记）、小核堆叠（参数-28%/非线性×2）
3. 数据增强首秀：3ep 短训负结果 → 10ep 长训对照（实测数字见 §5）——**增强的收益取决于训练时长与欠/过拟合状态**
4. **难点如实入账**：VGG 慢热、NiN 无 BN 难训——这两个"病"就是 03 项目 ResNet 要开的药方

**下一步**：`03_ResNet_CIFAR10` ★——家族重点。同结构有/无残差受控对比，亲眼看到残差连接如何让"加深度"从负担变成红利。